# ⼀、快速了解LangGraph

In [3]:
! pip install langgraph
##后⾯的案例中会⽤到langchain,所以同时也安装下langchain的依赖
! pip install langchain
! pip install langchain_community

Looking in indexes: https://pypi.doubanio.com/simple
Looking in indexes: https://pypi.doubanio.com/simple
Looking in indexes: https://pypi.doubanio.com/simple
  Using cached https://mirrors.cloud.tencent.com/pypi/packages/a0/f4/c67b0b3f1b9245e8d266f0f112c500d50e5b4e83cb6f3b71b6528104182a/requests-2.34.2-py3-none-any.whl (73 kB)
  Attempting uninstall: requests
    Found existing installation: requests 2.32.3
    Uninstalling requests-2.32.3:
      Successfully uninstalled requests-2.32.3


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
labelme 5.5.0 requires qtpy!=1.11.2, which is not installed.
labelme 5.5.0 requires scikit-image, which is not installed.
wandb 0.19.10 requires sentry-sdk>=2.0.0, which is not installed.
facenet-pytorch 2.6.0 requires numpy<2.0.0,>=1.24.0, but you have numpy 2.4.6 which is incompatible.
unstructured 0.22.29 requires charset-normalizer<4.0.0,>=3.4.4, but you have charset-normalizer 3.3.2 which is incompatible.
unstructured-client 0.44.0 requires pypdfium2>=5.6.0, but you have pypdfium2 4.30.1 which is incompatible.


# ⼆、构建聊天Agent智能体

要如何构建这样强⼤并且听话的Agent智能体呢？我们先从基础的⼤模型聊天开始。 
在LangChain中，提供了ChatModels的接⼝，⽤于访问⼤模型。我们可以通过调⽤ChatModels的接⼝来访问⼤模型。

In [1]:
from langchain_community.chat_models import ChatTongyi
# 构建阿⾥云百炼⼤模型客户端
model = ChatTongyi(
    api_key="sk-0e1e78cdb7034f8babefc27516eaedf2",
    base_url="https://api.aliyun.com/v1",
    model="qwen-max"
)
model.invoke("你是谁？能帮我解决什么问题？")

AIMessage(content='您好，我是Qwen，这是我的英文名，您也可以叫我通义千问。我是阿里云自主研发的超大规模语言模型，能够帮助您解答问题、创作文字，比如写故事、写公文、写邮件、写剧本等等，还能表达观点，玩游戏等。如果您有任何问题或需要帮助，请随时告诉我，我会尽力提供支持。', additional_kwargs={}, response_metadata={'model_name': 'qwen-max', 'finish_reason': 'stop', 'request_id': 'b2450dab-7323-974a-9abc-a98c687095f0', 'token_usage': {'input_tokens': 17, 'output_tokens': 75, 'prompt_tokens_details': {'cached_tokens': 0}, 'total_tokens': 92}}, id='lc_run--019e5352-f2c0-7bb3-80c7-33d34dc6016e-0', tool_calls=[], invalid_tool_calls=[])

⽽LangGraph只要将这个ChatModel进⾏简单的封装，就可以完成与⼤模型的交互。

In [2]:
from langgraph.prebuilt import create_react_agent
agent = create_react_agent(
    model=model,
    tools=[],
    prompt="You are a helpful assistant",
)
agent.invoke({"messages":[{"role":"user","content":"你是谁？能帮我解决什么问题？"}]})


C:\Users\Qinghe\AppData\Local\Temp\ipykernel_29404\2357108241.py:2: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


{'messages': [HumanMessage(content='你是谁？能帮我解决什么问题？', additional_kwargs={}, response_metadata={}, id='77b4d7e8-9918-4fc6-8ca5-2e7c2dac8709'),
  AIMessage(content='我是阿里云开发的一款超大规模语言模型，我叫通义千问。我可以生成各种类型的文本，如文章、故事、诗歌、故事等，并能够根据不同的场景和需求进行变换和扩展。此外，我还能够回答各种问题，提供帮助和解决方案。无论您需要创意写作、问题解答还是其他方面的帮助，我都会尽力满足您的需求。', additional_kwargs={}, response_metadata={'model_name': 'qwen-max', 'finish_reason': 'stop', 'request_id': '0b037ab8-0b2b-95c0-9b53-2b8c1196d67d', 'token_usage': {'input_tokens': 27, 'output_tokens': 77, 'prompt_tokens_details': {'cached_tokens': 0}, 'total_tokens': 104}}, id='lc_run--019e5353-107c-7341-9832-a9cbee30280e-0', tool_calls=[], invalid_tool_calls=[])]}

当然，Agent也⽀持常⽤的Stream流式输出的⽅式。

In [3]:
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "你是谁？能帮我解决什么问题？"}]},
    # stream_mode="messages",
):
    print(chunk)
    print()


{'agent': {'messages': [AIMessage(content='我是阿里云开发的一款超大规模语言模型，我叫通义千问。我可以生成各种类型的文本，如文章、故事、诗歌、故事等，并能够根据不同的场景和需求进行变换和扩展。此外，我还能够回答各种问题，提供帮助和解决方案。无论您需要创意写作、问题解答还是其他方面的帮助，我都会尽力满足您的需求。', additional_kwargs={}, response_metadata={'model_name': 'qwen-max', 'finish_reason': 'stop', 'request_id': '1c669106-c597-9905-a9a3-5b210d21b276', 'token_usage': {'input_tokens': 27, 'output_tokens': 77, 'prompt_tokens_details': {'cached_tokens': 0}, 'total_tokens': 104}}, id='lc_run--019e5353-2ad5-7712-93a5-19f414c16709-0', tool_calls=[], invalid_tool_calls=[])]}}



这⾥stream_mode有三种选项：
- updates : 流式输出每个⼯具调⽤的每个步骤。
- messages: 流式输出⼤语⾔模型回复的Token。
- values: ⼀次拿到所有的chunk。默认值。
- custom : ⾃定义输出。主要是可以在⼯具内部使⽤get_stream_writer获取输⼊流，添加⾃定义的内容。
关于流式输出的这⼏种选项，在后⾯结合Graph，会体现出更⼤的作⽤。

# 三、增加Tools⼯具调⽤

Tools⼯具机制是⼤语⾔模型中的⼀个重要机制，它可以让⼤模型调⽤外部⼯具，从⽽实现更加复杂的功能。通常，⼀个完整的⼯具调⽤流程需要以下⼏个步骤：
1. 客户端定义⼯具类，实现⼯具的功能。
2. 客户端请求⼤语⾔模型，带上问题以及⼯具的描述信息。
3. ⼤语⾔模型综合判断问题，并决定是否调⽤⼯具。
4. 如果⼤语⾔模型判断需要使⽤⼯具，就会向客户端返回⼀个带有tool_calls⼯具调⽤信息的AiMessage。
5. 客户端根据⼯具调⽤信息，调⽤⼯具，并将结果返回给⼤语⾔模型。
6. ⼤语⾔模型根据⼯具调⽤的结果，⽣成最终的回答。

使⽤LangChain，我们需要实现完整的流程。⽽使⽤LangGraph后，⼯具成了Agent的标配。只需要定义⼯具类，Agent中会⾃⾏完成⼯具调⽤的流程。

In [4]:
import datetime
from langgraph.prebuilt import create_react_agent

# 定义⼀个工具函数，获取当前日期
def get_current_date():
    """获取今天⽇期"""
    return datetime.datetime.today().strftime("%Y-%m-%d")
agent = create_react_agent(
    model=model,
    tools=[get_current_date],
    prompt="You are a helpful assistant",
)
agent.invoke({"messages":[{"role":"user","content":"今天是⼏⽉⼏号"}]})

C:\Users\Qinghe\AppData\Local\Temp\ipykernel_29404\2602189841.py:8: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


{'messages': [HumanMessage(content='今天是⼏⽉⼏号', additional_kwargs={}, response_metadata={}, id='4e1f04aa-67b7-4569-be02-be5bcff15605'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{}', 'name': 'get_current_date'}, 'id': 'call_b1d96f22f13747bfa704f6', 'index': 0, 'type': 'function'}]}, response_metadata={'model_name': 'qwen-max', 'finish_reason': 'tool_calls', 'request_id': '9af6acb8-b177-9454-b6c9-ee3e5f10de9c', 'token_usage': {'input_tokens': 227, 'output_tokens': 13, 'prompt_tokens_details': {'cached_tokens': 0}, 'total_tokens': 240}}, id='lc_run--019e5353-47ac-7151-b9a3-e5db648688b3-0', tool_calls=[{'name': 'get_current_date', 'args': {}, 'id': 'call_b1d96f22f13747bfa704f6', 'type': 'tool_call'}], invalid_tool_calls=[]),
  ToolMessage(content='2026-05-23', name='get_current_date', id='d21cb185-e077-4b42-a3a3-6559e5bb001a', tool_call_id='call_b1d96f22f13747bfa704f6'),
  AIMessage(content='今天是2026年5月23日。', additional_kwargs={}, response_metadata

从返回的结果就能看到，LangGraph的Agent完整的封装了⼯具调⽤的整个流程。⽽我们所需要关注的，只是构建出带有⼯具信息的Agent，然后把⽤户的问题交给Agent去处理就⾏了。

在定义⼯具时，除了可以从⼯具函数的注释中获取⼯具描述信息外，LangGraph也同样兼容了LangChain中使⽤@tool注解声明⼯具的⽅式。

如果⼯具执⾏时出错了，LangGraph也提供了主动处理异常信息的能⼒。


In [5]:
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode
# 定义⼯具 return_direct=True 表示直接返回⼯具的结果
@tool("devide_tool",return_direct=True)

# 定义⼀个工具函数，计算两个整数的除法
def devide(a : int,b : int) -> float:
    """计算两个整数的除法。
    Args:
        a (int): 除数
        b (int): 被除数"""
    # ⾃定义错误
    if b == 1:
        raise ValueError("除数不能为1")
    return a/b
print(devide.name)
print(devide.description)
print(devide.args)

# 定义⼯具调⽤错误处理函数
def handle_tool_error(error: Exception) -> str:
    """处理⼯具调⽤错误。
    Args:
        error (Exception): ⼯具调⽤错误"""
    if isinstance(error, ValueError):
        return "除数为1没有意义，请重新输⼊⼀个除数和被除数。"
    elif isinstance(error, ZeroDivisionError):
        return "除数不能为0，请重新输⼊⼀个除数和被除数。"
    return f"⼯具调⽤错误：{error}"

# 将工具和错误处理函数传⼊Agent中，Agent会在调⽤工具时捕获错误并调用错误处理函数
tool_node = ToolNode(
    [devide],
    handle_tool_errors=handle_tool_error
)
agent_with_error_handler = create_react_agent(
    model=model,
    tools=tool_node
)
result = agent_with_error_handler.invoke({"messages":[{"role":"user","content":"10除以5等于多少？"}]})
# 打印最后的返回结果
print(result["messages"][-1].content)
print(result)


devide_tool
计算两个整数的除法。
    Args:
        a (int): 除数
        b (int): 被除数
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


C:\Users\Qinghe\AppData\Local\Temp\ipykernel_29404\3610744680.py:36: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_with_error_handler = create_react_agent(


2.0
{'messages': [HumanMessage(content='10除以5等于多少？', additional_kwargs={}, response_metadata={}, id='302c0466-8198-4916-92a5-66f225bed115'), AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"a": 10, "b": 5}', 'name': 'devide_tool'}, 'id': 'call_bb9a0cbf125343479370cd', 'index': 0, 'type': 'function'}]}, response_metadata={'model_name': 'qwen-max', 'finish_reason': 'tool_calls', 'request_id': '7b5d130c-6b7a-9a60-bda1-d71937eaf0f8', 'token_usage': {'input_tokens': 277, 'output_tokens': 25, 'prompt_tokens_details': {'cached_tokens': 0}, 'total_tokens': 302}}, id='lc_run--019e5353-808e-7d02-807b-c5e7216e6c8d-0', tool_calls=[{'name': 'devide_tool', 'args': {'a': 10, 'b': 5}, 'id': 'call_bb9a0cbf125343479370cd', 'type': 'tool_call'}], invalid_tool_calls=[]), ToolMessage(content='2.0', name='devide_tool', id='d003ea5c-cc9d-45fc-bd1e-4c05bb372443', tool_call_id='call_bb9a0cbf125343479370cd')]}


# 四、增加消息记忆

对⼤模型的交互信息进⾏保存，这是实现多轮对话的关键。

![](./figures/1.png)

在LangChain中，我们需要⾃⾏定义ChatMessageHistory，并且⾃⾏保存每⼀轮的消息记录，然后在调⽤⼤模型时，将其作为参数传⼊。

LangGraph中，实现消息记录的流程，也完整的封装到了Agent当中。

LangGraph将消息记忆分为了短期记忆与⻓期记忆。

- 短期记忆是指Agent内部的记忆，⽤于当前对话中的历史记忆信息。LangGraph将他封装成CheckPoint。 
- ⻓期记忆是指Agent外部的记忆，⽤第三⽅存储⻓久的保存⽤户级别或者应⽤级别的聊天信息。LangGraph将它封装成Store.

![](./figures/2.png)

对于记忆管理，LangGraph的管理⽅式或许⽐具体实现更有参考价值。

在具体实现时，LangGraph都默认提供了InMemorySaver和InMemoryStore，也同样都可以转移到其他外部存储当中。不过，短期记忆通常是代表那些会话级别的⼩内存，⽽⻓期记忆通常是代表那些⽤户级别或者应⽤级别的⼤内存。

短期记忆的内存⽐较紧张，所以需要更频繁的清理内存，并对已有的消息记录进⾏总结，从⽽减少内存占⽤。⽽⻓期记忆的内存⽐较充⾜，所以不太需要频繁的清理内存。更需要关注的，是如何对已有的消息进⾏检索

## 4.1 短期记忆CheckPoint

在LangGraph的Agent中，只需要指定checkpointer属性，就可以实现短期记忆。具体传⼊的属性需要是BaseCheckpointSaver的⼦类。

LangGraph中默认提供了InMemorySaver，⽤于将短期记忆信息保存在内存中。当然，也可以采⽤Redis、SQLite等第三⽅存储来实现⻓期记忆。不过当前版本的LangGraph并没有提供具体的实现，需要⾃⾏实现。(如果不会写，交给AI)

另外，使⽤checkpointer时，需要制定⼀个单独的thread_id来区分不同的对话。


In [6]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.prebuilt import create_react_agent

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.prebuilt import create_react_agent

# 定义⼀个内存checkpointer，Agent会将对话历史保存在内存中
checkpointer = InMemorySaver()

# 定义⼀个工具函数，获取某个城市的天气
def get_weather(city: str) -> str:
    """获取某个城市的天气"""
    return f"城市：{city}，天气一直都是晴天！"

agent = create_react_agent(
    model=model,
    tools=[get_weather],
    checkpointer=checkpointer
)

# 通过configurable传⼊线程ID，保证同⼀线程的对话历史被正确关联
config = {
    "configurable": {
        "thread_id": "1"
    }
}

# 进⾏对话，Agent会将对话历史保存在内存中，并且在同⼀线程的后续对话中正确关联
cs_response = agent.invoke(
    {"messages": [{"role": "user", "content": "广州天气怎么样？"}]},
    config
)
print(cs_response)

# Continue the conversation using the same thread_id
bj_response = agent.invoke(
    {"messages": [{"role": "user", "content": "北京呢？"}]},
    config
)
bj_response


C:\Users\Qinghe\AppData\Local\Temp\ipykernel_29404\3025864574.py:15: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


{'messages': [HumanMessage(content='广州天气怎么样？', additional_kwargs={}, response_metadata={}, id='bd03a0de-1b04-4664-89ec-aae94b8a2dc5'), AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"city": "广州"}', 'name': 'get_weather'}, 'id': 'call_b66ed4d3139c49bb909467', 'index': 0, 'type': 'function'}]}, response_metadata={'model_name': 'qwen-max', 'finish_reason': 'tool_calls', 'request_id': 'e1c73f39-0664-9ea7-837f-2b31a2e83a10', 'token_usage': {'input_tokens': 230, 'output_tokens': 17, 'prompt_tokens_details': {'cached_tokens': 0}, 'total_tokens': 247}}, id='lc_run--019e5353-8f0f-7ca1-a2f2-63315d8359ac-0', tool_calls=[{'name': 'get_weather', 'args': {'city': '广州'}, 'id': 'call_b66ed4d3139c49bb909467', 'type': 'tool_call'}], invalid_tool_calls=[]), ToolMessage(content='城市：广州，天气一直都是晴天！', name='get_weather', id='3b606aa8-cf8c-41a9-b3f9-df8803ef1e6e', tool_call_id='call_b66ed4d3139c49bb909467'), AIMessage(content='广州的天气一直都是晴天！', additional_kwargs={}, response_m

{'messages': [HumanMessage(content='广州天气怎么样？', additional_kwargs={}, response_metadata={}, id='bd03a0de-1b04-4664-89ec-aae94b8a2dc5'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"city": "广州"}', 'name': 'get_weather'}, 'id': 'call_b66ed4d3139c49bb909467', 'index': 0, 'type': 'function'}]}, response_metadata={'model_name': 'qwen-max', 'finish_reason': 'tool_calls', 'request_id': 'e1c73f39-0664-9ea7-837f-2b31a2e83a10', 'token_usage': {'input_tokens': 230, 'output_tokens': 17, 'prompt_tokens_details': {'cached_tokens': 0}, 'total_tokens': 247}}, id='lc_run--019e5353-8f0f-7ca1-a2f2-63315d8359ac-0', tool_calls=[{'name': 'get_weather', 'args': {'city': '广州'}, 'id': 'call_b66ed4d3139c49bb909467', 'type': 'tool_call'}], invalid_tool_calls=[]),
  ToolMessage(content='城市：广州，天气一直都是晴天！', name='get_weather', id='3b606aa8-cf8c-41a9-b3f9-df8803ef1e6e', tool_call_id='call_b66ed4d3139c49bb909467'),
  AIMessage(content='广州的天气一直都是晴天！', additional_kwargs={}, resp

从结果可以看到，当前会话当中和⼤模型的每次交互记录，包括⼯具调⽤的信息，都保存在了短期记忆当中。当然，⽬前的实现是保存在内存中，所以程序结束后就释放了。⽣产环境中，LangGraph建议是保存到外部存储当中，例如数据库、⽂件系统等。这样每次启动程序时，都可以从外部存储中加载历史记录。

短期记忆通常认为是⽐较紧张的，所以需要定期做清理，防⽌历史消息过多。

LangGraph的Agent中，提供了⼀个pre_model_hook属性，可以在每次调⽤⼤模型之前触发。通过这个hook，就可以来定期管理短期记忆。

LangGraph中管理短期记忆的⽅法主要有两种：

- Summarization 总结：⽤⼤模型的⽅式，对短期记忆进⾏总结，然后再把总结的结果作为新的短期记忆。
- Trimming 删除：直接把短期记忆中最旧的消息删除掉。

LangGraph提供了SummarizationNode函数，⽤于使⽤⼤模型的⽅式对短期记忆进⾏总结

In [7]:
## LangGraph v1.0（2025年10月发布）已将 create_react_agent 标记为弃用,此版本代码无法演示

from langmem.short_term import SummarizationNode
from langchain_core.messages.utils import count_tokens_approximately
from langgraph.prebuilt import create_react_agent
from langgraph.prebuilt.chat_agent_executor import AgentState
from langgraph.checkpoint.memory import InMemorySaver
from typing import Any

# 使⽤⼤模型对历史信息进⾏总结
summarization_node = SummarizationNode(
    token_counter=count_tokens_approximately,
    model=model,
    max_tokens=384,
    max_summary_tokens=128,
    output_messages_key="llm_input_messages",
)

class State(AgentState):
    # 注意：这个状态管理的作⽤是为了能够保存上⼀次总结的结果。这样就可以防⽌每次调⽤⼤模型时，都要重新总结历史信息。
    # 这是⼀个⽐较常⻅的优化⽅式，因为⼤模型的调用是⽐较耗时的。
    context: dict[str, Any]
    
checkpointer = InMemorySaver()
agent = create_react_agent(
    model=model,
    tools=[],
    pre_model_hook=summarization_node,# 在调⽤模型之前先执⾏总结节点，对历史信息进⾏总结，减少模型的输入长度
    state_schema=State,
    checkpointer=checkpointer,
)




C:\Users\Qinghe\AppData\Local\Temp\ipykernel_29404\1201594943.py:23: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [10]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from typing_extensions import TypedDict, NotRequired
from typing import Any

# 1. 定义状态（TypedDict，新版不再支持 Pydantic BaseModel）
class State(TypedDict):
    # AgentState 默认包含 messages，这里只扩展自定义字段
    context: NotRequired[dict[str, Any]]

# 2. 创建 Agent
checkpointer = InMemorySaver()

agent = create_agent(
    model=model,                          # 你的主模型
    tools=[],                             # 工具列表
    system_prompt="You are a helpful assistant.",
    state_schema=State,
    checkpointer=checkpointer,
    middleware=[
        SummarizationMiddleware(
            model=model,                  # 用于生成总结的模型
            trigger=("tokens", 384),      # 历史消息超过 384 tokens 时触发总结
            keep=("messages", 2),          # 保留最近 2 条完整消息，更早的压缩为摘要
        )
    ],
)

另外，还提供了trim_messages函数，⽤于定期清理短期记忆。

In [ ]:
## LangChain 新版（v1.0+）已将该函数重命名为 create_agent，且参数结构也发生了变化——pre_model_hook 被移除，改为 middleware 机制，该例子不可展示

from langchain.agents import create_react_agent
from langchain_core.messages.utils import (
    trim_messages,
    count_tokens_approximately
)
# This function will be called every time before the node that calls LLM
def pre_model_hook(state):
    trimmed_messages = trim_messages(
        state["messages"], # 需要总结的消息列表
        strategy="last", # 保留最后的消息
        token_counter=count_tokens_approximately,
        max_tokens=384,
        start_on="human",
        end_on=("human", "tool"),
    )
    return {"llm_input_messages": trimmed_messages}
checkpointer = InMemorySaver()
agent = create_react_agent(
    model=model,
    tools=[],
    pre_model_hook=pre_model_hook,
    checkpointer=checkpointer,
)


ImportError: cannot import name 'create_react_agent' from 'langchain.agents' (c:\Users\Qinghe\.conda\envs\rag-agent-dev\Lib\site-packages\langchain\agents\__init__.py)

Middleware（中间件） 是 LangChain v1.0/LangGraph v1.0 引入的一种切面编程（AOP）机制，用来在 Agent 执行流程的特定节点插入自定义逻辑，而不需要修改 Agent 的核心代码。

In [12]:
#调用 LLM 前自动裁剪历史消息，控制 token 消耗
from langchain.agents import create_agent
from langchain.agents.middleware.types import AgentMiddleware
from langchain_core.messages.utils import trim_messages, count_tokens_approximately
from langgraph.checkpoint.memory import InMemorySaver
from typing import Any
from typing_extensions import TypedDict, NotRequired

# 1. 自定义中间件：包装 trim_messages 逻辑
class TrimMessagesMiddleware(AgentMiddleware):
    def before_model(self, state, runtime) -> dict[str, Any] | None:
        trimmed = trim_messages(
            state["messages"],           # 当前完整的历史消息列表
            strategy="last",             # 保留最后的消息，丢弃前面的
            token_counter=count_tokens_approximately,
            max_tokens=384,              # 裁剪后总长度不超过 384 tokens
            start_on="human",            # 裁剪边界必须从 human 消息开始
            end_on=("human", "tool"),    # 裁剪边界必须以 human 或 tool 消息结束
        )
        return {"messages": trimmed}      #返回一个 状态补丁（patch），只包含要修改的字段

# 2. 定义状态（新版只支持 TypedDict，不再支持 Pydantic）
class State(TypedDict):
    # 如有需要可扩展自定义字段，messages 由 AgentState 默认提供
    pass

# 3. 创建 Agent
checkpointer = InMemorySaver()

agent = create_agent(
    model=model,                          # 主 LLM（如 ChatOpenAI）
    tools=[],                             # ReAct 工具列表，空表示纯对话 Agent
    system_prompt="You are a helpful assistant.",
    state_schema=State,                   # 注册自定义状态结构
    checkpointer=checkpointer,            # 启用多轮记忆
    middleware=[TrimMessagesMiddleware()], # 在 LLM 调用前执行消息裁剪
)

实现了基础的短期记忆管理后，LangGraph还提供了状态管理机制，⽤于保存处理过程中的中间结果。⽽且，这些状态数据，还可以在Tools⼯具中使⽤。

In [14]:
from typing import Annotated
from langgraph.prebuilt import InjectedState, create_react_agent
from langgraph.prebuilt.chat_agent_executor import AgentState
from langchain_core.tools import tool

# 定义⼀个状态类，继承AgentState
class CustomState(AgentState):
    user_id: str

# 定义⼯具 return_direct=True 表示直接返回⼯具的结果
@tool(return_direct=True)

# 定义⼀个工具函数，查询⽤户信息
def get_user_info(
    state: Annotated[CustomState, InjectedState]
) -> str:
    """查询⽤户信息."""
    user_id = state["user_id"]
    return "user_123⽤户的姓名：Qinghe。" if user_id == "user_123" else "未知⽤户"

# 将工具和状态类传⼊Agent中，Agent会在调⽤工具时将状态类的实例传⼊工具函数中
agent = create_react_agent(
    model=model,
    tools=[get_user_info],
    state_schema=CustomState,
)

# 调⽤Agent，传⼊消息和状态信息，Agent会将状态信息注⼊到工具函数中
agent.invoke({
    "messages": "查询⽤户信息",
    "user_id": "user_123"
})

##这段代码实现了一个"有记忆的查询助手"：用户只需说"查询用户信息"，Agent 会自动从内部状态拿到 user_id，调用工具查数据库，并直接返回结果，不需要用户反复提供身份信息

C:\Users\Qinghe\AppData\Local\Temp\ipykernel_29404\3868349730.py:22: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


{'messages': [HumanMessage(content='查询⽤户信息', additional_kwargs={}, response_metadata={}, id='95eb98dd-6466-492f-8af9-7a43a8ff6a94'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{}', 'name': 'get_user_info'}, 'id': 'call_dd822375e523456492e022', 'index': 0, 'type': 'function'}]}, response_metadata={'model_name': 'qwen-max', 'finish_reason': 'tool_calls', 'request_id': '047bd76f-3f87-9674-90f9-cc3396ad836a', 'token_usage': {'input_tokens': 219, 'output_tokens': 13, 'prompt_tokens_details': {'cached_tokens': 0}, 'total_tokens': 232}}, id='lc_run--019e5362-a0ff-78b1-b812-6d74b8378f82-0', tool_calls=[{'name': 'get_user_info', 'args': {}, 'id': 'call_dd822375e523456492e022', 'type': 'tool_call'}], invalid_tool_calls=[]),
  ToolMessage(content='user_123⽤户的姓名：Qinghe。', name='get_user_info', id='4a8dde35-b5de-4b28-81fc-438db14a5198', tool_call_id='call_dd822375e523456492e022')],
 'user_id': 'user_123'}

## 4.2 长期记忆

⻓期记忆通常认为是⽐较充⾜的记忆空间，因此使⽤时，可以⽐短期记忆更加粗犷，不太需要实时关注内存空间⼤⼩。

⾄于使⽤⽅式，和短期记忆差不太多。主要是通过Agent的store属性指定⼀个  实现类就可以了。

与短期记忆最⼤的区别在于，短期记忆通过thread_id来区分不同的对话，⽽⻓期记忆则通过`namespace`来区分不同的命名空间。

In [20]:
from langchain_core.runnables import RunnableConfig
from langgraph.config import get_store
from langgraph.prebuilt import create_react_agent
from langgraph.store.memory import InMemoryStore
from langchain_core.tools import tool

# 定义⻓期存储
store = InMemoryStore()

# 添加⼀些测试数据。 users是命名空间，user_123是key，后⾯的JSON数据是value
store.put(
    ("users",),
    "user_123",
    {
        "name": "Qinghe",
        "age": "18",
    }
)

#定义⼯具
@tool(return_direct=True)
def get_user_info(config: RunnableConfig) -> str:
    """查找⽤户信息"""
    # 获取⻓期存储。获取到了后，这个存储组件可读也可写
    store = get_store()
    
    # store.put(


    #     ("users",),
    #     "user_456",
    #     {
    #         "name": "Qinghe",
    #         "age": "18",
    #     }
    # )

    # 获取配置中的⽤户ID
    user_id = config["configurable"].get("user_id")
    user_info = store.get(("users",), user_id)
    return str(user_info.value) if user_info else "Unknown user"
agent = create_react_agent(
    model=model,
    tools=[get_user_info],
    store=store,
    prompt="你是一个用户助手。当用户要求查找用户信息时，你必须调用 get_user_info 工具。用户ID已配置在系统中，不需要询问用户。"
)
# Run the agent
agent.invoke(
    {"messages": [{"role": "user", "content": "查找⽤户信息"}]},
    config={"configurable": {"user_id": "user_123"}}
)

C:\Users\Qinghe\AppData\Local\Temp\ipykernel_29404\843041739.py:42: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


{'messages': [HumanMessage(content='查找⽤户信息', additional_kwargs={}, response_metadata={}, id='eac0bca5-c011-452e-9aff-13f4d8f12d7e'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{}', 'name': 'get_user_info'}, 'id': 'call_bb92db6953954bc58c72f7', 'index': 0, 'type': 'function'}]}, response_metadata={'model_name': 'qwen-max', 'finish_reason': 'tool_calls', 'request_id': '2f276bbb-6139-9159-bbb9-35f479c7e2d5', 'token_usage': {'input_tokens': 255, 'output_tokens': 13, 'prompt_tokens_details': {'cached_tokens': 0}, 'total_tokens': 268}}, id='lc_run--019e53a7-4646-7221-a70f-605c1a62a99d-0', tool_calls=[{'name': 'get_user_info', 'args': {}, 'id': 'call_bb92db6953954bc58c72f7', 'type': 'tool_call'}], invalid_tool_calls=[]),
  ToolMessage(content="{'name': 'Qinghe', 'age': '18'}", name='get_user_info', id='3f96f1fc-0cba-44c9-a061-8411fcddabf3', tool_call_id='call_bb92db6953954bc58c72f7')]}

# 五、Human-in-the-loop⼈类监督

这也是LangGraph的Agent中⾮常核⼼的⼀个功能。

在Agent的⼯作过程中，有⼀个问题是⾮常致命的。就是Agent可以添加Tools⼯具，但是要不要调⽤⼯具，却完全是由Agent⾃⼰决定的。
这就会导致Agent在⾯对⼀些问题时，可能会出现错误的判断。

为了解决这个问题，LangGraph提供了Human-in-the-loop的功能。在Agent进⾏⼯具调⽤的过程中，允许⽤户进⾏监督。这就需要中断当前的执⾏任务，等待⽤户输⼊后，再重新恢复任务。

![](./figures/3.png)

在实现时，LangGraph提供了interruput()⽅法添加⼈类监督。监督时需要中断当前任务，所以通常是和stream流式⽅法配合使⽤。

In [21]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt
from langgraph.prebuilt import create_react_agent
from langchain_core.tools import tool
# An example of a sensitive tool that requires human review / approval

@tool(return_direct=True)
def book_hotel(hotel_name: str):
    """预定宾馆"""
    response = interrupt(f"正准备执⾏'book_hotel'⼯具预定宾馆，相关参数名： {{'hotel_name': {hotel_name}}}. "        "请选择OK，表示同意，或者选择edit，提出补充意⻅."
    )
    if response["type"] == "OK":
        pass
    elif response["type"] == "edit":
        hotel_name = response["args"]["hotel_name"]
    else:
        raise ValueError(f"Unknown response type: {response['type']}")
    return f"成功在 {hotel_name} 预定了⼀个房间."

checkpointer = InMemorySaver()

agent = create_react_agent(
    model=model,
    tools=[book_hotel],
    checkpointer=checkpointer,
)

config = {
    "configurable": {
    "thread_id": "1"
    }
}

for chunk in agent.stream({"messages": [{"role": "user", "content": "帮我在如家宾馆预定⼀个房间"}]},config
):
    print(chunk)
    print("\n")


C:\Users\Qinghe\AppData\Local\Temp\ipykernel_29404\4268371649.py:22: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


{'agent': {'messages': [AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"hotel_name": "如家宾馆"}', 'name': 'book_hotel'}, 'id': 'call_e1f6ab015f71453294de85', 'index': 0, 'type': 'function'}]}, response_metadata={'model_name': 'qwen-max', 'finish_reason': 'tool_calls', 'request_id': '1393cb32-d470-90ff-b897-de65a477c89a', 'token_usage': {'input_tokens': 241, 'output_tokens': 21, 'prompt_tokens_details': {'cached_tokens': 0}, 'total_tokens': 262}}, id='lc_run--019e53ab-26ed-7c71-b067-8f03ccf1e1b4-0', tool_calls=[{'name': 'book_hotel', 'args': {'hotel_name': '如家宾馆'}, 'id': 'call_e1f6ab015f71453294de85', 'type': 'tool_call'}], invalid_tool_calls=[])]}}


{'__interrupt__': (Interrupt(value="正准备执⾏'book_hotel'⼯具预定宾馆，相关参数名： {'hotel_name': 如家宾馆}. 请选择OK，表示同意，或者选择edit，提出补充意⻅.", id='72ed1765d4d4b7e8afcd8e45d0b919f4'),)}




执⾏完成后，会在book_hotel执⾏过程中，输出⼀个Interrupt响应，表示当前正在等待⽤户输⼊确认。

接下来，可以通过Agent提交⼀个Command请求，来继续完成之前的任务。

需要注意的是，在这个示例中，Agent只会⼀直等待⽤户输⼊。如果等待时间过⻓，后续请求就⽆法恢复了。

In [22]:
from langgraph.types import Command
for chunk in agent.stream(
    # Command(resume={"type": "OK"}),
    Command(resume={"type": "edit", "args": {"hotel_name": "三号宾馆"}}),    config
):
    print(chunk)
    print(chunk['tools']['messages'][-1].content)
    print("\n")

{'tools': {'messages': [ToolMessage(content='成功在 三号宾馆 预定了⼀个房间.', name='book_hotel', id='d143f88c-722d-4df8-9ca9-5de291674784', tool_call_id='call_e1f6ab015f71453294de85')]}}
成功在 三号宾馆 预定了⼀个房间.




# 六、多Agent应⽤构建

通常我们希望⼀个Agent能够专注⼲好⼀件事情。但是，如果是⾯对⼀些复杂的任务，我们可能需要多个Agent协同完成。例如⼀种典型的多Agent系统会是这样的：

![](./figures/4.png)

由⼀个Supervisor Agent对任务进⾏分发。然后，交由另⼀个Agent来处理具体的事情。这样，这多个Agent就可以成为⼀个同时处理多个任务的系统。

LangGraph中单独提供个⼀个langgraph-supervisor依赖库来实现这种类型的多Agent系统。

> 当然，LangGraph的核⼼是⽤Graph图的⽅式来实现多Agent协作。所以，这个库更多是作为基础了解。 

In [23]:
# 安装langgraph-supervisor依赖库
!pip install langgraph-supervisor --upgrade

Looking in indexes: https://pypi.doubanio.com/simple


In [25]:
from langgraph.prebuilt import create_react_agent
from langgraph_supervisor import create_supervisor
def book_hotel(hotel_name: str):
    """Book a hotel"""
    return f"Successfully booked a stay at {hotel_name}."

def book_flight(from_airport: str, to_airport: str):
    """Book a flight"""
    return f"Successfully booked a flight from {from_airport} to {to_airport}."
flight_assistant = create_react_agent(
    model=model,
    tools=[book_flight],
    prompt="You are a flight booking assistant",
    name="flight_assistant"
)
hotel_assistant = create_react_agent(
    model=model,
    tools=[book_hotel],
    prompt="You are a hotel booking assistant",
    name="hotel_assistant"
)
supervisor = create_supervisor(
    agents=[flight_assistant, hotel_assistant],
    model=model,
    prompt=(
        "You manage a hotel booking assistant and a"        "flight booking assistant. Assign work to them."    )
).compile()
for chunk in supervisor.stream(
    {
        "messages": [
            {
                "role": "user",
                "content":"book a flight from BOS to JFK and a stay at McKittrick Hotel"
                }
        ]
    }
):
    print(chunk)
    print("\n")


C:\Users\Qinghe\AppData\Local\Temp\ipykernel_29404\5902280.py:10: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  flight_assistant = create_react_agent(
C:\Users\Qinghe\AppData\Local\Temp\ipykernel_29404\5902280.py:16: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  hotel_assistant = create_react_agent(


{'supervisor': None}




ValueError: request_id: 3b8d533d-fb12-91c8-8960-ad68e6f2909e 
 status_code: 400 
 code: InvalidParameter 
 message: <400> InternalError.Algo.InvalidParameter: messages with role "tool" must be a response to a preceeding message with "tool_calls".